In [1]:
%pip install scikit-learn seaborn --quiet

import os
from pathlib import Path

BASE_DIR = Path.cwd()
if not (BASE_DIR / "df_allq.csv").exists():
    candidates = [
        Path("/Users/martateodoratrales/Desktop/AirPanel"),
        Path.cwd().parent,
    ]
    for candidate in candidates:
        if (candidate / "df_allq.csv").exists() and (candidate / "embedding_allq.csv").exists():
            BASE_DIR = candidate
            break

CACHE_DIR = Path("/tmp/airpanel_matplotlib_cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ["MPLCONFIGDIR"] = str(CACHE_DIR)

import matplotlib
matplotlib.use("Agg")
import pandas as pd
import numpy as np
import re
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams["figure.figsize"] = (10, 6)
sns.set_theme(style="whitegrid")
print(BASE_DIR)


Note: you may need to restart the kernel to use updated packages.
/Users/martateodoratrales/Desktop/AirPanel


In [2]:
df = pd.read_csv(BASE_DIR / "df_allq.csv")
emb = pd.read_csv(BASE_DIR / "embedding_allq.csv")
n = min(len(df), len(emb))
df = df.iloc[:n].copy().reset_index(drop=True)
emb = emb.iloc[:n].copy().reset_index(drop=True)
emb.columns = [f"emb_{i}" for i in range(emb.shape[1])]
full = pd.concat([df, emb], axis=1)
print(full.shape)
full.head(2)


(17600, 388)


,Unnamed: 0,panelist_id,Question,text,emb_0,emb_1,emb_2,emb_3,emb_4,emb_5,...,emb_374,emb_375,emb_376,emb_377,emb_378,emb_379,emb_380,emb_381,emb_382,emb_383
0,1,67b0d135d362f5886c2e5cfe,Q_bouygues_reactions_spontanees_1,"Ah ouais, marrant. C'est comme une série polic...",-0.111100,-0.011085,0.030964,-0.036024,0.001666,0.038861,...,0.038095,0.033253,-0.027515,0.009050,-0.095756,-0.035584,-0.011916,0.020798,0.045045,0.009730
1,2,67b0d135d362f5886c2e5cfe,Q_bouygues_memorisation_1,"Le délire scène de crime, avec le ruban de séc...",-0.041579,0.010726,-0.043304,-0.109654,0.047644,0.004194,...,0.020026,-0.008631,-0.021399,-0.005228,0.011391,-0.023041,-0.018921,0.083269,0.007809,-0.012313


In [3]:
def parse_question(q):
    if q.startswith("Q_bouygues_"):
        group = "Bouygues"
        family = q.replace("Q_bouygues_", "")
    elif q.startswith("Q_orange_"):
        group = "Orange"
        family = q.replace("Q_orange_", "")
    elif q.startswith("Q_comparaison_"):
        group = "Comparison"
        family = q.replace("Q_", "")
    else:
        group = "Other"
        family = q
    return group, family

parsed = full["Question"].apply(parse_question)
full["group"] = parsed.str[0]
full["family"] = parsed.str[1]
full["text"] = full["text"].fillna("").astype(str)
full["text_len"] = full["text"].str.len()
EMB_COLS = [c for c in full.columns if c.startswith("emb_")]
print(full["group"].value_counts().to_string())
full[["panelist_id", "Question", "group", "family", "text_len"]].head(6)


group
Bouygues      6400
Orange        6400
Comparison    4800


,panelist_id,Question,group,family,text_len
0,67b0d135d362f5886c2e5cfe,Q_bouygues_reactions_spontanees_1,Bouygues,reactions_spontanees_1,178
1,67b0d135d362f5886c2e5cfe,Q_bouygues_memorisation_1,Bouygues,memorisation_1,263
2,67b0d135d362f5886c2e5cfe,Q_bouygues_caractere_distinctif_1,Bouygues,caractere_distinctif_1,313
3,67b0d135d362f5886c2e5cfe,Q_bouygues_attractivite_1,Bouygues,attractivite_1,327
4,67b0d135d362f5886c2e5cfe,Q_bouygues_attractivite_2,Bouygues,attractivite_2,250
5,67b0d135d362f5886c2e5cfe,Q_bouygues_resonance_emotionnelle_1,Bouygues,resonance_emotionnelle_1,123


In [4]:
def l2_normalize(x):
    arr = np.asarray(x, dtype=float)
    norms = np.linalg.norm(arr, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    return arr / norms

def cosine_similarity_vec(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    na = np.linalg.norm(a)
    nb = np.linalg.norm(b)
    if na == 0 or nb == 0:
        return np.nan
    return float(np.dot(a, b) / (na * nb))

def centroid(df_part):
    return df_part[EMB_COLS].to_numpy(dtype=float).mean(axis=0)

def mean_distance_to_centroid(df_part, center):
    x = df_part[EMB_COLS].to_numpy(dtype=float)
    return float(np.linalg.norm(x - center, axis=1).mean())

def pick_output_dir(preferred):
    candidates = [
        preferred,
        Path.cwd() / "embedding_comparison_outputs",
        Path("/tmp/embedding_comparison_outputs"),
    ]
    for candidate in candidates:
        try:
            candidate.mkdir(parents=True, exist_ok=True)
            probe = candidate / ".write_test"
            probe.write_text("ok")
            probe.unlink()
            return candidate
        except Exception:
            continue
    raise PermissionError("No writable output directory found")

OUTPUT_DIR = pick_output_dir(BASE_DIR / "embedding_comparison_outputs")
print(OUTPUT_DIR)


/Users/martateodoratrales/Desktop/AirPanel/embedding_comparison_outputs


In [5]:
group_rows = []
centroids = {}
for group, part in full.groupby("group"):
    center = centroid(part)
    centroids[group] = center
    group_rows.append({
        "group": group,
        "n_answers": len(part),
        "n_panelists": part["panelist_id"].nunique(),
        "mean_text_len": part["text_len"].mean(),
        "mean_distance_to_centroid": mean_distance_to_centroid(part, center),
    })

group_summary = pd.DataFrame(group_rows).sort_values("group")

pairs = []
groups = sorted(centroids)
for i, g1 in enumerate(groups):
    for g2 in groups[i+1:]:
        pairs.append({
            "group_a": g1,
            "group_b": g2,
            "centroid_cosine_similarity": cosine_similarity_vec(centroids[g1], centroids[g2]),
            "centroid_euclidean_distance": float(np.linalg.norm(centroids[g1] - centroids[g2])),
        })

group_pairs = pd.DataFrame(pairs)
group_summary.to_csv(OUTPUT_DIR / "embedding_group_summary.csv", index=False)
group_pairs.to_csv(OUTPUT_DIR / "embedding_group_centroid_pairs.csv", index=False)
group_summary


,group,n_answers,n_panelists,mean_text_len,mean_distance_to_centroid
0,Bouygues,6400,800,229.275469,0.708373
1,Comparison,4800,800,221.576667,0.656980
2,Orange,6400,800,214.590625,0.721984


In [6]:
brand_only = full[full["group"].isin(["Bouygues", "Orange"])].copy()
family_counts = brand_only.groupby(["group", "family"]).size().unstack(fill_value=0)
common_families = sorted(set(family_counts.loc["Bouygues"][family_counts.loc["Bouygues"] > 0].index) & set(family_counts.loc["Orange"][family_counts.loc["Orange"] > 0].index))

family_rows = []
question_centroids = {}
for family in common_families:
    part_b = brand_only[(brand_only["group"] == "Bouygues") & (brand_only["family"] == family)]
    part_o = brand_only[(brand_only["group"] == "Orange") & (brand_only["family"] == family)]
    c_b = centroid(part_b)
    c_o = centroid(part_o)
    question_centroids[("Bouygues", family)] = c_b
    question_centroids[("Orange", family)] = c_o
    family_rows.append({
        "family": family,
        "n_bouygues": len(part_b),
        "n_orange": len(part_o),
        "cosine_similarity": cosine_similarity_vec(c_b, c_o),
        "euclidean_distance": float(np.linalg.norm(c_b - c_o)),
        "bouygues_dispersion": mean_distance_to_centroid(part_b, c_b),
        "orange_dispersion": mean_distance_to_centroid(part_o, c_o),
        "mean_len_bouygues": part_b["text_len"].mean(),
        "mean_len_orange": part_o["text_len"].mean(),
    })

family_summary = pd.DataFrame(family_rows).sort_values("cosine_similarity")
family_summary.to_csv(OUTPUT_DIR / "embedding_family_comparison.csv", index=False)
family_summary


,family,n_bouygues,n_orange,cosine_similarity,euclidean_distance,bouygues_dispersion,orange_dispersion,mean_len_bouygues,mean_len_orange
6,reactions_spontanees_1,800,800,0.788425,0.524819,0.570693,0.603715,213.99750,179.66000
5,memorisation_1,800,800,0.829885,0.442643,0.645053,0.650008,266.75875,255.49375
2,caractere_distinctif_1,800,800,0.851203,0.427213,0.597281,0.645086,247.01000,231.68875
3,image_1,800,800,0.862193,0.420695,0.586094,0.603317,218.21000,217.07125
4,intention_achat_1,800,800,0.867880,0.379234,0.690937,0.655012,228.12000,235.57875
7,resonance_emotionnelle_1,800,800,0.883783,0.367508,0.633864,0.653168,176.99500,173.22500
0,attractivite_1,800,800,0.921046,0.290213,0.675678,0.686473,235.70500,226.42375
1,attractivite_2,800,800,0.929636,0.279917,0.668059,0.658959,247.40750,197.58375


In [7]:
pair_rows = []
for family in common_families:
    part_b = brand_only[(brand_only["group"] == "Bouygues") & (brand_only["family"] == family)][["panelist_id"] + EMB_COLS]
    part_o = brand_only[(brand_only["group"] == "Orange") & (brand_only["family"] == family)][["panelist_id"] + EMB_COLS]
    merged = part_b.merge(part_o, on="panelist_id", suffixes=("_b", "_o"))
    emb_b = merged[[f"{c}_b" for c in EMB_COLS]].to_numpy(dtype=float)
    emb_o = merged[[f"{c}_o" for c in EMB_COLS]].to_numpy(dtype=float)
    num = np.sum(emb_b * emb_o, axis=1)
    den = np.linalg.norm(emb_b, axis=1) * np.linalg.norm(emb_o, axis=1)
    cos = np.divide(num, den, out=np.full_like(num, np.nan, dtype=float), where=den != 0)
    pair_rows.append({
        "family": family,
        "n_pairs": len(merged),
        "mean_panelist_cosine": float(np.nanmean(cos)),
        "median_panelist_cosine": float(np.nanmedian(cos)),
        "std_panelist_cosine": float(np.nanstd(cos)),
    })

paired_similarity = pd.DataFrame(pair_rows).sort_values("mean_panelist_cosine")
paired_similarity.to_csv(OUTPUT_DIR / "embedding_panelist_paired_similarity.csv", index=False)
paired_similarity


,family,n_pairs,mean_panelist_cosine,median_panelist_cosine,std_panelist_cosine
4,intention_achat_1,800,0.474694,0.478314,0.081782
5,memorisation_1,800,0.485752,0.486864,0.071846
0,attractivite_1,800,0.494864,0.493825,0.066695
7,resonance_emotionnelle_1,800,0.511194,0.511321,0.071385
6,reactions_spontanees_1,800,0.516255,0.516425,0.061271
1,attractivite_2,800,0.519343,0.520261,0.073867
2,caractere_distinctif_1,800,0.522645,0.523390,0.080752
3,image_1,800,0.562696,0.567457,0.075417


In [8]:
centroid_rows = []
labels = []
vectors = []
for (group, family), vec in question_centroids.items():
    labels.append(f"{group} | {family}")
    vectors.append(vec)
    centroid_rows.append({"group": group, "family": family})

centroid_meta = pd.DataFrame(centroid_rows)
centroid_matrix = np.vstack(vectors)
sim_matrix = l2_normalize(centroid_matrix) @ l2_normalize(centroid_matrix).T
sim_df = pd.DataFrame(sim_matrix, index=labels, columns=labels)
sim_df.to_csv(OUTPUT_DIR / "embedding_question_centroid_similarity_matrix.csv")
sim_df.iloc[:6, :6]


,Bouygues | attractivite_1,Orange | attractivite_1,Bouygues | attractivite_2,Orange | attractivite_2,Bouygues | caractere_distinctif_1,Orange | caractere_distinctif_1
Bouygues | attractivite_1,1.000000,0.921046,0.902840,0.891666,0.847607,0.870923
Orange | attractivite_1,0.921046,1.000000,0.865951,0.850697,0.804282,0.855059
Bouygues | attractivite_2,0.902840,0.865951,1.000000,0.929636,0.815434,0.835238
Orange | attractivite_2,0.891666,0.850697,0.929636,1.000000,0.838036,0.876162
Bouygues | caractere_distinctif_1,0.847607,0.804282,0.815434,0.838036,1.000000,0.851203
Orange | caractere_distinctif_1,0.870923,0.855059,0.835238,0.876162,0.851203,1.000000


In [9]:
plt.figure(figsize=(14, 10))
sns.heatmap(sim_df, cmap="viridis", vmin=0, vmax=1)
plt.title("Question centroid cosine similarity")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "embedding_question_centroid_similarity_heatmap.png", dpi=300)
plt.show()


/var/folders/k9/89sz9rmn25s78pr1yz7ytdz40000gn/T/ipykernel_57479/3634510634.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [10]:
# sample_per_group = 1200
# sampled = (
#     full.groupby("group", group_keys=False)
#     .apply(lambda x: x.sample(min(sample_per_group, len(x)), random_state=42))
#     .reset_index(drop=True)
# )

# x = sampled[EMB_COLS].to_numpy(dtype=float)
# pca = PCA(n_components=2, random_state=42)
# coords = pca.fit_transform(x)
# sampled["pc1"] = coords[:, 0]
# sampled["pc2"] = coords[:, 1]

# plt.figure(figsize=(11, 8))
# sns.scatterplot(
#     data=sampled,
#     x="pc1",
#     y="pc2",
#     hue="group",
#     style="group",
#     alpha=0.6,
#     s=35,
# )
# plt.title("PCA of answer embeddings")
# plt.tight_layout()
# plt.savefig(OUTPUT_DIR / "embedding_pca_answers_by_group.png", dpi=300)
# plt.show()


In [11]:
comp_rows = []
comparison_only = full[full["group"] == "Comparison"].copy()
for family in sorted(comparison_only["family"].unique()):
    part_c = comparison_only[comparison_only["family"] == family]
    c_c = centroid(part_c)
    row = {
        "comparison_family": family,
        "n_answers": len(part_c),
    }
    for group in ["Bouygues", "Orange", "Comparison"]:
        row[f"cosine_to_{group.lower()}_centroid"] = cosine_similarity_vec(c_c, centroids[group])
    comp_rows.append(row)

comparison_summary = pd.DataFrame(comp_rows).sort_values("comparison_family")
comparison_summary.to_csv(OUTPUT_DIR / "embedding_comparison_questions_vs_group_centroids.csv", index=False)
comparison_summary


,comparison_family,n_answers,cosine_to_bouygues_centroid,cosine_to_orange_centroid,cosine_to_comparison_centroid
0,comparaison_1,800,0.736884,0.804463,0.952069
1,comparaison_2,800,0.737505,0.782095,0.946559
2,comparaison_3,800,0.782521,0.821059,0.918267
3,comparaison_4,800,0.685622,0.748713,0.906650
4,comparaison_5,800,0.734175,0.814556,0.953562
5,comparaison_6,800,0.665879,0.745176,0.934232


In [12]:
full.to_csv(OUTPUT_DIR / "df_allq_with_embeddings_metadata.csv", index=False)
print("saved:")
for p in sorted(OUTPUT_DIR.iterdir()):
    print(p.name)


saved:
df_allq_with_embeddings_metadata.csv
embedding_comparison_questions_vs_group_centroids.csv
embedding_family_comparison.csv
embedding_group_centroid_pairs.csv
embedding_group_summary.csv
embedding_panelist_paired_similarity.csv
embedding_question_centroid_similarity_heatmap.png
embedding_question_centroid_similarity_matrix.csv


# Embeddings Outputs

- `df_allq_with_embeddings_metadata.csv`  
  The full answer-level dataset: original respondent answers, question labels, brand/comparison group, question family, text length, and all embedding dimensions.

- `embedding_comparison_questions_vs_group_centroids.csv`  
  For each comparison question, shows how close its average embedding is to the overall Bouygues, Orange, and Comparison semantic centroids.

- `embedding_family_comparison.csv`  
  Compares matched Bouygues vs Orange question families (for example image, memorisation, intention d’achat) using centroid similarity and distance.

- `embedding_group_centroid_pairs.csv`  
  Pairwise comparison of the main groups (`Bouygues`, `Orange`, `Comparison`) based on their average embedding vectors.

- `embedding_group_summary.csv`  
  High-level summary by group: number of answers, number of panelists, mean text length, and average distance to the group centroid.

- `embedding_panelist_paired_similarity.csv`  
  For each matched question family, measures how similar each panelist’s Bouygues answer embedding is to their Orange answer embedding, then summarizes those similarities.

- `embedding_question_centroid_similarity_heatmap.png`  
  Visual heatmap of semantic similarity between all question-family centroids.

- `embedding_question_centroid_similarity_matrix.csv`  
  The numeric version of the heatmap: cosine similarities between all question-family centroids.
